In [ ]:
# CELL 1: Imports and Load Data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for better visuals
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("Loading data for EDA and Feature Engineering...\n")

# Load the data from previous notebook
df = pd.read_csv("data/processed/final_dataset.csv")

print(f"Dataset loaded: {df.shape[0]} stocks × {df.shape[1]} columns")
print(f"Number of high-growth stocks (3x+ in 5 years): {df['high_growth'].sum()} ({df['high_growth'].mean():.2%})")

display(df.head())

In [ ]:
# CELL 2: Summary of High-Growth Stocks

print("Top 21 High-Growth Stocks (3x+ return)\n")

high_growth_stocks = df[df['high_growth'] == True].sort_values('return_5y', ascending=False)

display(high_growth_stocks[['ticker', 'company_name', 'sector', 'return_5y']].round(2).head(21))

# Summary statistics
print("\n5-Year Return Statistics:")
print(df['return_5y'].describe().round(3))

In [ ]:
# CELL 3: Visual Exploratory Data Analysis

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Distribution of 5-Year Returns
sns.histplot(df['return_5y'], bins=50, kde=True, ax=axes[0,0])
axes[0,0].axvline(x=2.0, color='red', linestyle='--', label='3x Return Threshold')
axes[0,0].set_title('Distribution of 5-Year Returns')
axes[0,0].set_xlabel('5-Year Return (2.0 = 300% return)')
axes[0,0].legend()

# 2. High-Growth Rate by Sector
sector_perf = df.groupby('sector')['high_growth'].mean().sort_values(ascending=False)
sector_perf.plot(kind='bar', ax=axes[0,1])
axes[0,1].set_title('High-Growth Rate by Sector')
axes[0,1].set_ylabel('Percentage of 3x+ Stocks')
axes[0,1].tick_params(axis='x', rotation=45)

# 3. Revenue Growth vs 5-Year Return
sns.scatterplot(data=df, x='revenue_growth', y='return_5y', hue='high_growth', 
                ax=axes[1,0], s=80)
axes[1,0].set_title('Revenue Growth vs 5-Year Return')
axes[1,0].set_xlabel('Revenue Growth')
axes[1,0].axhline(y=2.0, color='red', linestyle='--')

# 4. PEG Ratio vs Return
sns.boxplot(data=df, x='high_growth', y='peg_ratio', ax=axes[1,1])
axes[1,1].set_title('PEG Ratio: High-Growth vs Normal Stocks')
axes[1,1].set_xlabel('Is High Growth (3x+)')

plt.tight_layout()
plt.show()

In [ ]:
# CELL 4: Feature Engineering

print("🔧 Creating new features...\n")

# 1. Growth Features
df['revenue_growth_strong'] = (df['revenue_growth'] > 0.20).astype(int)
df['earnings_growth_strong'] = (df['earnings_growth'] > 0.25).astype(int)
df['growth_score'] = df['revenue_growth'] + df['earnings_growth']

# 2. Valuation Features
df['peg_ratio_clean'] = df['peg_ratio'].clip(upper=5)
df['pe_category'] = pd.cut(df['forward_pe'], bins=[0, 15, 25, 40, np.inf], 
                           labels=['Cheap', 'Fair', 'Expensive', 'Very Expensive'])

# 3. Quality & Profitability
df['roe_strong'] = (df['return_on_equity'] > 0.15).astype(int)
df['low_debt'] = (df['debt_to_equity'] < 1.0).astype(int)

# 4. Risk Features
df['beta_category'] = pd.cut(df['beta'], bins=[0, 0.8, 1.2, 2, np.inf],
                             labels=['Low', 'Moderate', 'High', 'Very High'])

# 5. Size Feature
df['log_market_cap'] = np.log1p(df['market_cap'])

# 6. Combined Quality Score
df['quality_score'] = (
    df['roe_strong'] * 2.0 +
    df['low_debt'] * 1.5 +
    df['revenue_growth_strong'] * 2.0 +
    (df['peg_ratio_clean'] < 2).astype(int) * 2.0
)

print("New features created:")
print(df[['growth_score', 'quality_score', 'roe_strong', 'low_debt', 'revenue_growth_strong']].head())

In [ ]:
# CELL 5: Feature Correlation with High Growth

numeric_features = ['market_cap', 'forward_pe', 'peg_ratio_clean', 'revenue_growth', 
                    'earnings_growth', 'return_on_equity', 'debt_to_equity', 'beta',
                    'log_market_cap', 'growth_score', 'quality_score', 
                    'revenue_growth_strong', 'earnings_growth_strong', 'roe_strong', 'low_debt']

corr_with_target = df[numeric_features + ['high_growth']].corr()['high_growth'].sort_values(ascending=False)

print("📈 Top Features Correlated with High-Growth (3x+):")
print(corr_with_target.head(12))

# Visualization
plt.figure(figsize=(10, 8))
sns.heatmap(df[numeric_features + ['high_growth']].corr(), cmap='coolwarm', center=0, annot=True, fmt='.2f')
plt.title('Correlation Matrix with High-Growth Target')
plt.tight_layout()
plt.show()

In [ ]:
# CELL 6: Save Final Engineered Dataset

final_columns = ['ticker', 'company_name', 'sector', 'industry', 'market_cap', 'forward_pe',
                 'peg_ratio_clean', 'revenue_growth', 'earnings_growth', 'return_on_equity',
                 'debt_to_equity', 'beta', 'log_market_cap', 'growth_score', 'quality_score',
                 'revenue_growth_strong', 'earnings_growth_strong', 'roe_strong', 'low_debt',
                 'high_growth', 'return_5y']

final_df = df[final_columns].copy()

final_df.to_csv("data/processed/final_engineered_dataset.csv", index=False)
final_df.to_pickle("data/processed/final_engineered_dataset.pkl")

print("Final engineered dataset saved successfully!")
print(f"Shape: {final_df.shape}")
print("Ready for Machine Learning!")